# 05 — Model Teks + Ensemble 3 Model (Submission Final)

> Notebook ini adalah **hasil run asli di Kaggle** (CPU) lengkap dengan output-nya. **Submission final** berasal dari notebook ini.

**Peran dalam workflow:** melatih model teks dari hasil OCR (TF-IDF kata dan karakter + Logistic Regression), lalu menggabungkan tiga model (Donut + DiT + Teks) dengan rata-rata geometrik berbobot.

| | |
|---|---|
| Input | Data kompetisi + output notebook 02, 03 (Version 3), dan 04 |
| Output | `submission.csv`, `text_val_probs.npy`, `text_test_probs.npy` |
| Durasi | sekitar 1 jam (CPU) |
| Hasil | Teks saja **0,8010** · Ensemble 3 model holdout **0,9216** · Public **0,91677** |

Kode modul: `src/text_model.py`, `src/ensemble.py`, `src/train_text.py`, `src/make_ensemble.py`, `configs/text.yaml`.

# RVL-CDIP — OCR Text Model + 3-Way Ensemble (Donut + DiT + Text)

**Inputs (Add Input), 4 in total:**
1. The competition data
2. The Donut notebook output (`val_predictions.csv`, `val_probs.npy`, `test_probs.npy`)
3. The DiT notebook output (`dit_val_ids.csv`, `dit_val_probs.npy`, `dit_test_probs.npy`)
4. The docTR OCR notebook output (`ocr_text.csv`)

**Settings:** Accelerator **None** (CPU only: no GPU quota used). Internet not needed.

**Pipeline:** TF-IDF (words + character n-grams) → logistic regression, trained on train minus Donut's holdout →
score it on the holdout → tune 3-way ensemble weights on the holdout → refit the text model on all train → `submission.csv`.

**Decision rule:** the 2-way Donut+DiT ensemble scored **0.9107** on the holdout. Submit the 3-way result only if it beats that clearly (about +0.5 points).

## 1. Config

In [1]:
import re, time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

C_GRID          = [1.0, 4.0, 16.0]   # LR regularization; picked on the holdout
MIN_WORDS       = 5                  # pages with fewer OCR words get a uniform text vote (= no influence)
WEIGHT_STEP     = 0.05
REFIT_FULL      = True               # refit the text model on all train (incl. holdout) for test predictions
OUT = Path("/kaggle/working")

## 2. Locate inputs

In [2]:
def find_file(name):
    hits = sorted(Path("/kaggle/input").rglob(name))
    assert len(hits) == 1, f"Expected exactly one {name} under /kaggle/input, found {len(hits)}: {hits}"
    return hits[0]

ROOT = find_file("train.csv").parent
train_df   = pd.read_csv(ROOT / "train.csv")
test_df    = pd.read_csv(ROOT / "test.csv")
sample_sub = pd.read_csv(ROOT / "sample_submission.csv")

cls_map = train_df.groupby("label")["class_name"].agg(lambda s: s.unique().tolist())
CLASSES = [cls_map[i][0] for i in sorted(cls_map.index)]
NUM_CLASSES = len(CLASSES)

donut_val_ids = pd.read_csv(find_file("val_predictions.csv"))["id"].values
donut_val     = np.load(find_file("val_probs.npy"))
donut_test    = np.load(find_file("test_probs.npy"))
dit_val_ids   = pd.read_csv(find_file("dit_val_ids.csv"))["id"].values
dit_val       = np.load(find_file("dit_val_probs.npy"))
dit_test      = np.load(find_file("dit_test_probs.npy"))
ocr = pd.read_csv(find_file("ocr_text.csv"))

assert donut_val.shape == (len(donut_val_ids), NUM_CLASSES) and dit_val.shape == (len(dit_val_ids), NUM_CLASSES)
assert donut_test.shape == dit_test.shape == (len(test_df), NUM_CLASSES)
print("Data root:", ROOT)
print(f"Holdout: {len(donut_val_ids)} | OCR pages: {len(ocr)} | test: {len(test_df)}")

Data root: /kaggle/input/competitions/document-image-classification-sd234504
Holdout: 3201 | OCR pages: 39997 | test: 7994


## 3. Align everything by id
The holdout is Donut's. Both models' probabilities are re-indexed to the same row order, and test rows follow `test.csv`.

In [3]:
holdout_ids = donut_val_ids
P_donut_val = pd.DataFrame(donut_val, index=donut_val_ids).loc[holdout_ids].values
P_dit_val   = pd.DataFrame(dit_val,   index=dit_val_ids).loc[holdout_ids].values
P_donut_te, P_dit_te = donut_test, dit_test                      # both already in test.csv order

ocr["text"] = ocr["text"].fillna("")
ocr = ocr.set_index("id")
missing = set(train_df["id"]) - set(ocr.index) | set(test_df["id"]) - set(ocr.index)
assert not missing, f"{len(missing)} ids have no OCR record"

lab = train_df.set_index("id")["label"]
va_ids = pd.Index(holdout_ids)
tr_ids = pd.Index(train_df["id"][~train_df["id"].isin(set(holdout_ids))])
all_tr_ids = pd.Index(train_df["id"])
te_ids = pd.Index(test_df["id"])
y_tr, y_va, y_all = lab.loc[tr_ids].values, lab.loc[va_ids].values, lab.loc[all_tr_ids].values

print("Sanity - Donut holdout acc:", round(accuracy_score(y_va, P_donut_val.argmax(1)), 4),
      "| DiT holdout acc:", round(accuracy_score(y_va, P_dit_val.argmax(1)), 4))

Sanity - Donut holdout acc: 0.8919 | DiT holdout acc: 0.8907


## 4. Text features
- Lowercase; every digit becomes `0`, so Bates numbers, dates and amounts collapse into a few shared patterns.
- **Word 1–2-grams** catch keywords ("invoice", "period covered").
- **Character 3–5-grams** survive OCR misspellings ("labornt**ory**" still shares fragments with "laboratory").
- `min_df=3` drops tokens seen on fewer than 3 pages, which removes most OCR garbage.
- Two page-level features: word count and mean OCR confidence (near-empty pages are themselves a signal, e.g. file_folder).

In [4]:
def norm(t):
    return re.sub(r"\d", "0", str(t).lower())

def texts(ids):
    return [norm(t) for t in ocr.loc[ids, "text"]]

def meta(ids):
    n = ocr.loc[ids, "n_words"].values.astype(float)
    c = ocr.loc[ids, "mean_conf"].values.astype(float)
    return sp.csr_matrix(np.column_stack([np.log1p(n) / np.log1p(1000), c]))

def build_vectorizers():
    word = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=3, max_features=200_000,
                           sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=300_000,
                           sublinear_tf=True)
    return word, char

def featurize(fit_ids, other_ids_list):
    word, char = build_vectorizers()
    t_fit = texts(fit_ids)
    X_fit = sp.hstack([word.fit_transform(t_fit), char.fit_transform(t_fit), meta(fit_ids)]).tocsr()
    others = []
    for ids in other_ids_list:
        t = texts(ids)
        others.append(sp.hstack([word.transform(t), char.transform(t), meta(ids)]).tocsr())
    return X_fit, others

t0 = time.time()
X_tr, (X_va,) = featurize(tr_ids, [va_ids])
print(f"Features: {X_tr.shape[1]:,} | built in {(time.time()-t0)/60:.1f} min")

Features: 500,002 | built in 1.7 min


## 5. Train the text model and pick C on the holdout

In [5]:
def fit_lr(X, y, C):
    return LogisticRegression(C=C, max_iter=1000, solver="lbfgs").fit(X, y)

def gate(P, ids):
    """Uniform probabilities on near-empty pages: adds the same amount to every class, so no influence on the ensemble."""
    P = P.copy()
    P[ocr.loc[ids, "n_words"].values < MIN_WORDS] = 1.0 / NUM_CLASSES
    return P

results = {}
for C in C_GRID:
    t0 = time.time()
    clf = fit_lr(X_tr, y_tr, C)
    P = clf.predict_proba(X_va)
    results[C] = (accuracy_score(y_va, P.argmax(1)), P)
    print(f"C={C:>5}: text-only holdout acc {results[C][0]:.4f} | {(time.time()-t0)/60:.1f} min")

BEST_C = max(results, key=lambda c: results[c][0])
P_text_val_raw = results[BEST_C][1]
P_text_val = gate(P_text_val_raw, va_ids)
print("Best C:", BEST_C)

print(classification_report(y_va, P_text_val_raw.argmax(1), labels=list(range(NUM_CLASSES)),
                            target_names=CLASSES, digits=3, zero_division=0))

C=  1.0: text-only holdout acc 0.7929 | 10.6 min
C=  4.0: text-only holdout acc 0.8010 | 12.8 min
C= 16.0: text-only holdout acc 0.7988 | 14.3 min
Best C: 4.0
                        precision    recall  f1-score   support

         advertisement      0.762     0.716     0.738       201
                budget      0.756     0.741     0.749       201
                 email      0.989     0.930     0.959       201
           file_folder      0.501     0.842     0.628       202
                  form      0.764     0.692     0.726       201
           handwritten      0.725     0.857     0.786       203
               invoice      0.871     0.818     0.844       198
                letter      0.848     0.853     0.851       197
                  memo      0.882     0.824     0.852       199
          news_article      0.851     0.726     0.784       197
          presentation      0.635     0.533     0.579       199
         questionnaire      0.925     0.821     0.870       195
        

## 6. Ensemble weights on the holdout
Two ways of combining are tried:
- **mean:** weighted average of probabilities.
- **geo:** weighted average of log-probabilities. This is often better when the models' confidence levels differ (logistic regression vs. label-smoothed networks).

The 2-way rows (text weight 0) should reproduce the earlier 0.9107. The weights are chosen on the holdout, so its accuracy is slightly optimistic; the leaderboard is the real check.

In [6]:
def combine(Ps, ws, mode):
    if mode == "mean":
        return sum(w * P for w, P in zip(ws, Ps))
    return sum(w * np.log(np.clip(P, 1e-7, 1.0)) for w, P in zip(ws, Ps))

steps = np.round(np.arange(0, 1 + 1e-9, WEIGHT_STEP), 2)
rows = []
for mode in ["mean", "geo"]:
    for w_donut in steps:
        for w_text in steps:
            w_dit = round(1 - w_donut - w_text, 2)
            if w_dit < -1e-9:
                continue
            S = combine([P_donut_val, P_dit_val, P_text_val], [w_donut, w_dit, w_text], mode)
            rows.append((mode, w_donut, w_dit, w_text, accuracy_score(y_va, S.argmax(1))))
grid = pd.DataFrame(rows, columns=["mode", "w_donut", "w_dit", "w_text", "val_acc"])

best2 = grid[grid["w_text"] == 0].sort_values("val_acc", ascending=False).iloc[0]
best3 = grid.sort_values("val_acc", ascending=False).iloc[0]
print("Top 10 combinations:")
print(grid.sort_values("val_acc", ascending=False).head(10).to_string(index=False))
print(f"\nBest 2-way (Donut+DiT): {best2.val_acc:.4f}  [{best2['mode']}, donut {best2.w_donut}, dit {best2.w_dit}]")
print(f"Best 3-way (+Text):     {best3.val_acc:.4f}  [{best3['mode']}, donut {best3.w_donut}, dit {best3.w_dit}, text {best3.w_text}]")
print(f"Gain from text: {100*(best3.val_acc - best2.val_acc):+.2f} points")

Top 10 combinations:
mode  w_donut  w_dit  w_text  val_acc
 geo     0.40   0.25    0.35 0.921587
 geo     0.45   0.20    0.35 0.921587
 geo     0.45   0.25    0.30 0.920962
 geo     0.40   0.20    0.40 0.920650
 geo     0.35   0.20    0.45 0.920337
 geo     0.50   0.20    0.30 0.920025
 geo     0.50   0.25    0.25 0.920025
 geo     0.35   0.25    0.40 0.920025
 geo     0.30   0.25    0.45 0.919088
 geo     0.40   0.30    0.30 0.919088

Best 2-way (Donut+DiT): 0.9103  [geo, donut 0.6, dit 0.4]
Best 3-way (+Text):     0.9216  [geo, donut 0.4, dit 0.25, text 0.35]
Gain from text: +1.12 points


## 7. Where text helps: per-class F1, 2-way vs 3-way

In [7]:
S2 = combine([P_donut_val, P_dit_val, P_text_val], [best2.w_donut, best2.w_dit, 0.0], best2["mode"])
S3 = combine([P_donut_val, P_dit_val, P_text_val], [best3.w_donut, best3.w_dit, best3.w_text], best3["mode"])
f2 = f1_score(y_va, S2.argmax(1), average=None, labels=list(range(NUM_CLASSES)))
f3 = f1_score(y_va, S3.argmax(1), average=None, labels=list(range(NUM_CLASSES)))
f_text = f1_score(y_va, P_text_val_raw.argmax(1), average=None, labels=list(range(NUM_CLASSES)))
cmp = pd.DataFrame({"text_only": f_text, "2way": f2, "3way": f3, "delta": f3 - f2}, index=CLASSES).round(3)
print(cmp.sort_values("delta", ascending=False).to_string())

cm = confusion_matrix(y_va, S3.argmax(1), labels=list(range(NUM_CLASSES)))
pairs = [(cm[i, j], CLASSES[i], CLASSES[j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j and cm[i, j] > 0]
print("\nTop 3-way confusions (count, true -> predicted):")
for c, a, b in sorted(pairs, reverse=True)[:10]:
    print(f"  {c:4d}  {a} -> {b}")

                        text_only   2way   3way  delta
scientific_report           0.741  0.817  0.862  0.045
form                        0.726  0.782  0.809  0.027
invoice                     0.844  0.918  0.940  0.021
questionnaire               0.870  0.907  0.923  0.016
advertisement               0.738  0.905  0.920  0.015
news_article                0.784  0.915  0.930  0.015
budget                      0.749  0.877  0.892  0.015
presentation                0.579  0.850  0.859  0.010
letter                      0.851  0.912  0.921  0.009
resume                      0.975  0.980  0.988  0.007
file_folder                 0.628  0.927  0.929  0.003
memo                        0.852  0.944  0.946  0.002
scientific_publication      0.871  0.938  0.940  0.002
email                       0.959  0.995  0.995  0.000
specification               0.908  0.940  0.940 -0.000
handwritten                 0.786  0.958  0.954 -0.005

Top 3-way confusions (count, true -> predicted):
    11  scienti

## 8. Test predictions and submission
The text model is refit on all 32k training pages (cheap on CPU) with the chosen C. Donut and DiT are reused as they are.

In [8]:
t0 = time.time()
fit_ids = all_tr_ids if REFIT_FULL else tr_ids
y_fit = y_all if REFIT_FULL else y_tr
X_fit, (X_te,) = featurize(fit_ids, [te_ids])
clf = fit_lr(X_fit, y_fit, BEST_C)
P_text_te = gate(clf.predict_proba(X_te), te_ids)
print(f"Refit on {len(fit_ids)} pages in {(time.time()-t0)/60:.1f} min")

np.save(OUT / "text_val_probs.npy", P_text_val_raw)
pd.DataFrame({"id": va_ids}).to_csv(OUT / "text_val_ids.csv", index=False)
np.save(OUT / "text_test_probs.npy", P_text_te)

S_te = combine([P_donut_te, P_dit_te, P_text_te], [best3.w_donut, best3.w_dit, best3.w_text], best3["mode"])
pred_df = pd.DataFrame({"id": te_ids, "label": S_te.argmax(1)})
sub = sample_sub[["id"]].merge(pred_df, on="id", how="left")
assert sub["label"].notna().all() and len(sub) == len(sample_sub), "Submission is missing ids"
sub["label"] = sub["label"].astype(int)
sub.to_csv(OUT / "submission.csv", index=False)

prev = combine([P_donut_te, P_dit_te], [0.65, 0.35], "mean").argmax(1)
print(f"Predictions changed vs. the 0.89836 submission: {(prev != S_te.argmax(1)).sum()} of {len(sub)}")
print(f"Holdout: 2-way {best2.val_acc:.4f} -> 3-way {best3.val_acc:.4f}. "
      + ("Submit submission.csv." if best3.val_acc - best2.val_acc >= 0.005
         else "Gain is under 0.5 points: submitting is optional; the text model adds little."))

Refit on 32003 pages in 15.5 min
Predictions changed vs. the 0.89836 submission: 267 of 7994
Holdout: 2-way 0.9103 -> 3-way 0.9216. Submit submission.csv.
